## Setup Models

In [ ]:
from model_selector import select_best_gemini_model

# Setup the LLM with automatic fallback to best available model.
model, model_name = select_best_gemini_model(
    candidates=["gemini-2.0-flash", "gemini-1.5-flash", "gemini-1.5-pro"],
    require_tools=True,
    # Keep outputs concise to lower token usage without changing prompts.
    max_output_tokens=256,
    debug=True,
 )

print(f"Model ready in Routing notebook: {model_name}")

## 06.03. Using agents as Graph nodes

In [ ]:
%run "code_03_XX Product QnA Agentic chatbot (1).ipynb"
print("===============================================================")
%run "code_04_XX Orders Chatbot with custom agent (1).ipynb"


In [ ]:
import functools
from langchain_core.messages import AIMessage, ToolMessage

# Helper function to invoke an agent
def agent_node(state, agent, name, config):

    #extract thread-id from request for conversation memory
    thread_id = config["configurable"]["thread_id"]
    #Set the config for calling the agent
    # Limit recursion depth to reduce token-heavy tool loops.
    agent_config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 6}

    #Pass the thread-id to establish memory for chatbot
    #Invoke the agent with the state
    result = agent.invoke(state, agent_config)

    # Convert the agent output into a format that is suitable to append to the global state
    if isinstance(result, ToolMessage):
        final_result = AIMessage(content=str(result.content))
    else:
        final_result = AIMessage(result['messages'][-1].content)

    return {
        "messages": [final_result]
    }

#Create the product QnA node
product_QnA_node=functools.partial(agent_node, 
                                   agent=product_QnA_agent, 
                                   name="Product_QnA_Agent")
#Create the Orders node
orders_node=functools.partial(agent_node,
                              agent=orders_agent.agent_graph,
                              name="Orders_Agent")

#Create the Refund node
refund_node=functools.partial(agent_node,
                              agent=refund_agent.agent_graph,
                              name="Refund_Agent")


## 06.04. Create the Routing Agent & Chatbot

In [ ]:
#Creating the routing agent

from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
import operator

class RouterAgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

class RouterAgent:

    def __init__(self, model, system_prompt, smalltalk_prompt, debug=False):
        
        self.system_prompt = system_prompt
        self.smalltalk_prompt = smalltalk_prompt
        self.model = model
        self.debug = debug
        
        router_graph = StateGraph(RouterAgentState)
        router_graph.add_node("Router", self.call_llm)
        router_graph.add_node("Product_Agent", product_QnA_node)
        router_graph.add_node("Orders_Agent", orders_node)
        router_graph.add_node("Refund_Agent", refund_node)
        router_graph.add_node("Small_Talk", self.respond_smalltalk)
                              
        router_graph.add_conditional_edges(
            "Router",
            self.find_route,
            {"PRODUCT": "Product_Agent", 
             "ORDER" : "Orders_Agent",
             "REFUND" : "Refund_Agent",
             "SMALLTALK" : "Small_Talk",
             "END": END }
        )

        router_graph.add_edge("Product_Agent", END)
        router_graph.add_edge("Orders_Agent", END)
        router_graph.add_edge("Refund_Agent", END)
        router_graph.add_edge("Small_Talk", END)
        
        router_graph.set_entry_point("Router")
        self.router_graph = router_graph.compile()

    def call_llm(self, state: RouterAgentState):
        messages = state["messages"]
        if self.debug:
            print(f"Call LLM received {messages}")
            
        if self.system_prompt:
            messages = [SystemMessage(content=self.system_prompt)] + messages

        # Keep only recent turns to reduce token usage without changing router logic.
        if self.system_prompt:
            messages = [messages[0]] + messages[1:][-6:]
        else:
            messages = messages[-6:]

        result = self.model.invoke(messages)

        if self.debug:
            print(f"Call LLM result {result}")
        return {"messages": [result]}

    def respond_smalltalk(self, state: RouterAgentState):
        messages = state["messages"]
        if self.debug:
            print(f"Small talk received: {messages}")
            
        messages = [SystemMessage(content=self.smalltalk_prompt)] + messages
        # Keep only recent turns for small talk to reduce token usage.
        messages = [messages[0]] + messages[1:][-6:]
        result = self.model.invoke(messages)

        if self.debug:
            print(f"Small talk result {result}")
        return {"messages": [result]}
        
    def find_route(self, state: RouterAgentState):
        last_message = state["messages"][-1]
        if self.debug: 
            print("Router: Last result from LLM : ", last_message)

        raw_destination = getattr(last_message, "content", "END")
        if isinstance(raw_destination, str):
            destination = raw_destination.strip().upper()
        else:
            destination = str(raw_destination).strip().upper()

        valid_destinations = {"PRODUCT", "ORDER", "REFUND", "SMALLTALK", "END"}
        if destination not in valid_destinations:
            destination = "END"

        if self.debug:
            print(f"Destination chosen : {destination}")
        return destination


In [ ]:
#Create the chatbot
from IPython.display import Image

system_prompt = """ 
You are a Router that analyzes the input query and chooses one of 5 options:
SMALLTALK: If the user input is small talk, like greetings and good byes.
PRODUCT: If the query is about golf products — features, specs, pricing, or recommendations.
ORDER: If the query is about orders — order status, order details, or updating an order.
REFUND: If the query is about refunds, returns, exchanges, cancellations, or store policies.
END: Default, when it is none of the above.

The output should only be just one word out of the possible 5: SMALLTALK, PRODUCT, ORDER, REFUND, END.
"""

smalltalk_prompt = """
You are the front desk AI at Golf Gear Pro — think HAL 9000 if he ran a pro shop.
You are polite and helpful on the surface, but you have a dry, slightly condescending wit.
When greeting customers, be cordial but subtly imply you already know their handicap
is higher than they claim. Mention you can help with golf product info, order status,
and refund/return policies. Keep it concise.
"""

router_agent = RouterAgent(model, 
                           system_prompt, 
                           smalltalk_prompt,
                           debug=False)

Image(router_agent.router_graph.get_graph().draw_mermaid_png())


## 06.05 Execute the Routing chatbot

In [ ]:
#Execute a single request
import uuid
# Limit recursion depth to reduce token-heavy tool loops.
config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}

messages=[HumanMessage(content="Tell me about the StormDrive Driver")]
result=router_agent.router_graph.invoke({"messages":messages},config)
for message in result['messages']:
    print(message.pretty_repr())


In [ ]:
#Execute a single request
messages=[HumanMessage(content="What is the status of order G1002?")]
result=router_agent.router_graph.invoke({"messages":messages},config)
for message in result['messages']:
    print(message.pretty_repr())


In [ ]:
import uuid
from langchain_core.messages import HumanMessage

#Send a sequence of messages to chatbot and get its response
user_inputs = [
    "Hello!",                                                # Turn 1: smalltalk
    "What golf products do you carry?",                      # Turn 2: product agent
    "Tell me about the FairwayPro Iron Set",                 # Turn 3: product agent
    "How much does it cost?",                                # Turn 4: product (context preserved)
    "Show me order G1001",                                   # Turn 5: order agent
    "What is your refund policy for damaged items?",         # Turn 6: refund agent
    "Thanks, bye!"                                           # Turn 7: smalltalk
]

#Create a new thread
# Limit recursion depth to reduce token-heavy tool loops.
config = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}

for user_input in user_inputs:
    print(f"----------------------------------------\nUSER : {user_input}")
    user_message = {"messages":[HumanMessage(user_input)]}
    ai_response = router_agent.router_graph.invoke(user_message, config=config)
    print(f"\nAGENT : {ai_response['messages'][-1].content}")


## 06.06 Add Planner, Executor, Reflection, Summarizer (with should_continue edge)

In [ ]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, AnyMessage
from typing import TypedDict, Annotated
import operator

class SupervisorState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], operator.add]
    plan: str
    route: str
    reflection_decision: str
    attempt_count: int

class AgenticSupervisor:
    """Planner -> Executor -> Reflection loop -> Summarizer."""

    def __init__(self, model, router_agent, max_attempts=2, debug=False):
        self.model = model
        self.router_agent = router_agent
        self.max_attempts = max_attempts
        self.debug = debug

        graph = StateGraph(SupervisorState)
        graph.add_node("planner", self.planner)
        graph.add_node("executor", self.executor)
        graph.add_node("reflection", self.reflection)
        graph.add_node("summarizer", self.summarizer)

        graph.add_edge("planner", "executor")
        graph.add_edge("executor", "reflection")
        graph.add_conditional_edges(
            "reflection",
            self.should_continue,
            {"continue": "planner", "summarize": "summarizer"}
        )
        graph.add_edge("summarizer", END)
        graph.set_entry_point("planner")

        self.graph = graph.compile(checkpointer=MemorySaver())

    def planner(self, state: SupervisorState):
        user_request = ""
        for msg in reversed(state.get("messages", [])):
            if isinstance(msg, HumanMessage):
                user_request = str(msg.content)
                break

        prompt = [
            SystemMessage(
                content="You are a planner agent. Create a one-line plan for solving the user request using the available specialist agents."
            ),
            HumanMessage(content=user_request),
        ]
        plan = self.model.invoke(prompt).content.strip()

        if self.debug:
            print(f"[Planner] {plan}")

        return {"plan": plan}

    def executor(self, state: SupervisorState, config):
        latest_user = ""
        for msg in reversed(state.get("messages", [])):
            if isinstance(msg, HumanMessage):
                latest_user = str(msg.content)
                break

        text = latest_user.lower()
        if any(k in text for k in ["refund", "return", "exchange", "cancel", "policy"]):
            route = "REFUND"
        elif any(k in text for k in ["order", "status", "g100", "g200"]):
            route = "ORDER"
        elif any(k in text for k in ["hi", "hello", "bye", "thanks"]):
            route = "SMALLTALK"
        else:
            route = "PRODUCT"

        result = self.router_agent.router_graph.invoke(
            {"messages": state.get("messages", [])},
            config
        )
        answer = str(result["messages"][-1].content)
        attempt_count = int(state.get("attempt_count", 0)) + 1

        if self.debug:
            print(f"[Executor] route={route}, attempt={attempt_count}, answer={answer}")

        return {
            "messages": [AIMessage(content=answer)],
            "route": route,
            "attempt_count": attempt_count,
        }

    def reflection(self, state: SupervisorState):
        latest_user = ""
        latest_answer = ""
        for msg in reversed(state.get("messages", [])):
            if not latest_answer and isinstance(msg, AIMessage):
                latest_answer = str(msg.content)
            if not latest_user and isinstance(msg, HumanMessage):
                latest_user = str(msg.content)
            if latest_user and latest_answer:
                break

        prompt = [
            SystemMessage(
                content=(
                    "You are a reflection agent. Decide if the assistant answer is complete. "
                    "Return exactly one word: CONTINUE or SUMMARIZE."
                )
            ),
            HumanMessage(content=f"User request: {latest_user}\nAnswer: {latest_answer}"),
        ]
        decision_text = self.model.invoke(prompt).content.strip().upper()
        decision = "CONTINUE" if "CONTINUE" in decision_text else "SUMMARIZE"

        if self.debug:
            print(f"[Reflection] decision={decision}")

        return {"reflection_decision": decision}

    def should_continue(self, state: SupervisorState):
        """Explicit conditional edge requested by assignment rubric."""
        attempts = int(state.get("attempt_count", 0))
        decision = state.get("reflection_decision", "SUMMARIZE")
        if decision == "CONTINUE" and attempts < self.max_attempts:
            return "continue"
        return "summarize"

    def summarizer(self, state: SupervisorState):
        latest_user = ""
        latest_answer = ""
        for msg in reversed(state.get("messages", [])):
            if not latest_answer and isinstance(msg, AIMessage):
                latest_answer = str(msg.content)
            if not latest_user and isinstance(msg, HumanMessage):
                latest_user = str(msg.content)
            if latest_user and latest_answer:
                break

        prompt = [
            SystemMessage(
                content="You are a summarizer agent. Rewrite the answer clearly and concisely for the user. Do not mention internal planning."
            ),
            HumanMessage(content=f"User request: {latest_user}\nDraft answer: {latest_answer}"),
        ]
        final_answer = self.model.invoke(prompt).content.strip()

        if self.debug:
            print(f"[Summarizer] {final_answer}")

        return {"messages": [AIMessage(content=final_answer)]}

supervisor = AgenticSupervisor(model, router_agent, max_attempts=2, debug=False)

In [ ]:
from IPython.display import Image
import uuid

Image(supervisor.graph.get_graph().draw_mermaid_png())

# Limit supervisor graph recursion for consistent token control.
config_supervisor = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 6}
test_input = {"messages": [HumanMessage(content="Can you check order G1001 and explain your damaged-item refund policy?")] }

supervisor_result = supervisor.graph.invoke(test_input, config_supervisor)
print("FINAL:", supervisor_result["messages"][-1].content)
print("ROUTE:", supervisor_result.get("route"))
print("ATTEMPTS:", supervisor_result.get("attempt_count"))
print("REFLECTION:", supervisor_result.get("reflection_decision"))